# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    # TODO: loop up to `tries` times

    # loop through the allowed number of attempts
    for attempt in range(tries):
        try:
            # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
            r = requests.get(BASE, params=params, timeout=10)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            # TODO:   raise on a status that is not in RETRY_STATUSES
            if r.status_code not in RETRY_STATUSES: # 400 or 404
                r.raise_for_status()

            # TODO:   otherwise sleep (increasing each time) and try again

            delay = (attempt + 1) * 2 # each time longer w/ pause
            time.sleep(delay)

        # if last try gets network error
        except requests.RequestException as e:
            if attempt == tries - 1:
                raise e
            delay = (attempt + 1) * 2
            time.sleep(delay)

    # TODO: raise if every attempt failed
    raise RuntimeError("all attempts failed")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

try:
    raw_wx = get_weather(start='2026-09-05', end='2026-10-03')
    wx_filtered = raw_wx[raw_wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()

# if it does fail then use fallback
except Exception as e:
    print(f"failed to fetch weather data: {e}")
    used_fallback = True
    wx_filtered = pd.DataFrame() # initialize as empty df

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
if used_fallback or wx_filtered.empty:
    used_fallback = True
    wx = FALLBACK.copy()
else: # not fail
    wx = wx_filtered.copy()

# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO

wx['date'] = pd.to_datetime(wx['time'])

# rename existing columns
wx = wx.rename(columns={
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx = wx[['date', 'temp_max', 'precip_mm', 'source']] # final frame

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO

# find the row with the maximum precipitation
wetter_row = wx.loc[wx['precip_mm'].idxmax()]
# find the row with the minimum precipitation
drier_row = wx.loc[wx['precip_mm'].idxmin()]

# extract values for simple calculations
wetter_date = wetter_row['date'].strftime('%Y-%m-%d')
drier_date = drier_row['date'].strftime('%Y-%m-%d')
difference = wetter_row['precip_mm'] - drier_row['precip_mm']

# print the descriptive sentence for the vendor
print(f"{wetter_date} was wetter than {drier_date} by {difference:.1f} mm of precipitation.")

2026-10-03 was wetter than 2026-09-05 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO

# define vars
endpoint = BASE
lat, lon = 38.03, -78.51
start_date, end_date = '2026-09-05', '2026-10-03'
pull_time = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
data_type = 'fallback sample' if used_fallback else 'real data (open-meteo)'

print(f"api endpoint: {endpoint}")
print(f"coordinates: latitude {lat}, longitude {lon}")
print(f"date range: from {start_date} to {end_date}")
print(f"retrieved at: {pull_time}")
print(f"data source type: {data_type}")

api endpoint: https://archive-api.open-meteo.com/v1/archive
coordinates: latitude 38.03, longitude -78.51
date range: from 2026-09-05 to 2026-10-03
retrieved at: 2026-10-09 17:08:22
data source type: real data (open-meteo)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

if the function returned an empty dataframe on failure instead of raising an error, a left join of weather onto sales would result in null values for temperature and precipitation for those days.

when charting rain against revenue, standard plotting libraries often ignore or drop rows with null values entirely. the chart would simply show the remaining days where weather data existed, hiding the missing data. because there are no error messages or visible blank spots on a continuous line chart or scatter plot, nobody would notice that critical weather data was missing, leading to incomplete or incorrect analysis.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch attempt

# fetch orlando weather data for the same target dates
try:
    orlando_raw = get_weather(start='2026-09-05', end='2026-10-03', lat=28.54, lon=-81.38)
    orlando_wx = orlando_raw[orlando_raw['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    orlando_source = 'open-meteo'

except Exception as e:
    print("using fallback for orlando")

    orlando_wx = pd.DataFrame({
        'time': ['2026-09-05', '2026-10-03'],
        'temperature_2m_max': [32.5, 29.0],
        'precipitation_sum': [4.2, 0.0],
    })
    orlando_source = 'fallback sample'

# formatting orlando dataset
orlando_wx['date'] = pd.to_datetime(orlando_wx['time'])
orlando_wx = orlando_wx.rename(columns={
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

orlando_wx['source'] = orlando_source
orlando_wx['city'] = 'orlando'
orlando_wx = orlando_wx[['date', 'temp_max', 'precip_mm', 'source', 'city']]

